In [1]:
# imports
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

# PT3 imports
from distfit import distfit                          # fits statistical distributions to data
from scipy.stats import pearson3, norm              # PT3 distribution and normal distribution tools

# sci-kit learn imports
from sklearn.linear_model import ElasticNet, SGDRegressor  # the model we're training
from sklearn.model_selection import GridSearchCV    # cross-validated hyperparameter search
from sklearn.metrics import mean_squared_error      # computes MSE
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

# pytorch imports
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# for data transfer
import pickle


In [2]:
# load data

with open('../data/processed/kmeans.pkl', 'rb') as f:
    data = pickle.load(f)

X_train_clustered = data['X_train_clustered']
X_train_scaled = data['X_train_scaled'].drop(columns=['Cluster'], errors='ignore')
X_test_scaled = data['X_test_scaled']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
k_means_labels = data['K_means_labels']


In [3]:
# show data

X_train_scaled.head()

,HSBASHHD,HSFD001S,HSHO001S,HSHF001S,HSTR001S,HSHC001S,HSPC001S,HSRE001S,HSRO001S,HSTA018S,...,ECYGEN1GEN,ECYGEN2GEN,ECYGEN3GEN,ECYTCAHPOP,ECYTCACIT,ECYTCA_U18,ECYTCA_18P,ECYNCANCIT,ECYNCA_U18,ECYNCA_18P
0,2.096885,1.924531,1.521558,1.919040,1.829394,2.207040,1.950844,2.051773,0.913626,1.965526,...,-0.228224,-0.121147,4.314653,1.727888,2.247607,2.242034,2.181706,-0.248492,-0.197283,-0.255273
1,-0.332897,-0.416294,-0.376221,-0.305056,0.000908,-0.388333,-0.318336,-0.293081,-0.566061,-0.265945,...,-0.520022,-0.405742,0.054563,-0.397312,-0.345664,-0.138669,-0.404826,-0.415133,-0.352709,-0.421172
2,-0.368225,-0.380159,-0.381707,-0.348902,-0.490942,-0.514085,-0.318584,-0.254495,0.029062,-0.317163,...,-0.375272,-0.179227,-0.136139,-0.336145,-0.329678,-0.200775,-0.363070,-0.248492,-0.274996,-0.237498
3,0.511050,0.525045,0.682765,0.458800,0.522289,0.285856,0.419810,0.668126,-0.139953,0.733696,...,-0.078878,-0.051450,1.383013,0.547221,0.645463,0.875717,0.548595,0.104396,0.139473,0.094301
4,0.063562,-0.177893,-0.182823,-0.271947,-0.167056,-0.100117,-0.179068,-0.200501,-0.232490,-0.145046,...,-0.225926,-0.173419,0.006080,-0.179671,-0.203567,-0.359488,-0.145013,-0.057345,-0.171379,-0.030124


In [4]:
# ============================================================
# STEP 3a-i â€” Fit PT3 distribution and transform y â†’ z_normal
# ============================================================

# --- Fit PT3 to y_train ---

# distfit tries many distributions; we restrict it to just 'pearson3'
# so it finds the best-fitting PT3 parameters on the training set only.
dfit = distfit(distr='pearson3')
dfit.fit_transform(y_train.values if hasattr(y_train, 'values') else y_train)

# Extract the three PT3 parameters: skewness (skew), location (loc), scale (scale).
# These describe the shape of your data's distribution.
pt3_params = dfit.model['params']   # returns (skew, loc, scale)
skew, loc, scale = pt3_params

print(f"PT3 params â€” skew: {skew:.4f}, loc: {loc:.4f}, scale: {scale:.4f}")


# --- Helper: transform y â†’ z_normal ---

def transform_to_znormal(y, skew, loc, scale, eps=1e-6):
    """
    Converts raw y values into z_normal:
      1. Evaluate the PT3 CDF at each y value â†’ gives a probability in [0,1].
      2. Clip away from 0 and 1 to prevent the next step returning Â±infinity.
      3. Apply the inverse standard-normal CDF (PPF) â†’ gives z values ~ N(0,1).
    """
    y_arr = np.array(y)

    # Step 1: PT3 CDF â€” "what percentile is each value?"
    cdf_vals = pearson3.cdf(y_arr, skew, loc=loc, scale=scale)

    # Step 2: Clip to (eps, 1-eps) so norm.ppf never hits -inf or +inf
    cdf_vals = np.clip(cdf_vals, eps, 1 - eps)

    # Step 3: Normal PPF â€” "what z-score corresponds to this percentile?"
    z = norm.ppf(cdf_vals)
    return z


# --- Transform training and test targets ---

# We fit PT3 only on y_train; we apply the same parameters to y_test
# (no "peeking" at the test distribution).
z_train = transform_to_znormal(y_train, skew, loc, scale)
z_test  = transform_to_znormal(y_test,  skew, loc, scale)

print(f"z_train â€” mean: {z_train.mean():.3f}, std: {z_train.std():.3f}")

[10-04-2026 15:19:05] [distfit.distfit] [INFO] fit
[10-04-2026 15:19:05] [distfit.distfit] [INFO] transform
[10-04-2026 15:19:06] [distfit.distfit] [INFO] [pearson3] [0.66 sec] [RSS: 18.8303] [loc=0.257 scale=0.049]
[10-04-2026 15:19:06] [distfit.distfit] [INFO] [pearson3] [0.66 sec] [RSS: 18.8303] [loc=0.257 scale=0.049]
[10-04-2026 15:19:06] [distfit.distfit] [INFO] Compute confidence intervals [parametric]


PT3 params â€” skew: 0.9501, loc: 0.2570, scale: 0.0489
z_train â€” mean: -0.004, std: 0.984


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

class ElasticNetModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return self.linear(x)

def train_model(alpha, l1_ratio, X, z, epochs=50, lr=1e-3):
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    z_t = torch.tensor(z, dtype=torch.float32).view(-1, 1).to(device)
    g = torch.Generator()
    g.manual_seed(42)
    loader = DataLoader(TensorDataset(X_t, z_t), batch_size=4096, shuffle=True, generator=g)
    model = ElasticNetModel(X.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            mse = torch.mean((preds - yb) ** 2)
            l1_pen = sum(p.abs().sum() for p in model.parameters())
            l2_pen = sum((p**2).sum() for p in model.parameters())
            loss = mse + alpha * (l1_ratio * l1_pen + (1 - l1_ratio) * l2_pen)
            loss.backward()
            optimizer.step()
    return model

alphas    = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
l1_ratios = [0.1, 0.25, 0.5, 0.75, 1.0]

X_train_np = X_train_scaled.values  # numpy array for KFold indexing

# Move all training data to GPU once before the loops
X_train_gpu = torch.tensor(X_train_np, dtype=torch.float32).to(device)
z_train_gpu = torch.tensor(z_train,    dtype=torch.float32).to(device)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_r2, best_params = -np.inf, None

for l1 in tqdm(l1_ratios, desc="l1_ratio"):
    for alpha in tqdm(alphas, desc=f"alpha (l1={l1})", leave=False):
        fold_r2s = []
        for train_idx, val_idx in kf.split(X_train_np):
            train_idx_t = torch.tensor(train_idx, dtype=torch.long).to(device)
            val_idx_t   = torch.tensor(val_idx,   dtype=torch.long).to(device)

            Xf_tr_t  = X_train_gpu[train_idx_t]
            zf_tr_t  = z_train_gpu[train_idx_t].view(-1, 1)
            Xf_val_t = X_train_gpu[val_idx_t]
            zf_val   = z_train[val_idx]   # keep on CPU for r2_score

            model = ElasticNetModel(Xf_tr_t.shape[1]).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
            n = Xf_tr_t.shape[0]
            for _ in range(50):
                perm = torch.randperm(n, device=device)  # shuffle on GPU
                for start in range(0, n, 4096):
                    idx = perm[start:start+4096]
                    xb, yb = Xf_tr_t[idx], zf_tr_t[idx]
                    optimizer.zero_grad()
                    preds = model(xb)
                    mse = torch.mean((preds - yb) ** 2)
                    l1_pen = sum(p.abs().sum() for p in model.parameters())
                    l2_pen = sum((p**2).sum() for p in model.parameters())
                    loss = mse + alpha * (l1 * l1_pen + (1 - l1) * l2_pen)
                    loss.backward()
                    optimizer.step()
            with torch.no_grad():
                val_preds = model(Xf_val_t).cpu().numpy().flatten()
            fold_r2s.append(r2_score(zf_val, val_preds))
        mean_r2 = np.mean(fold_r2s)
        if mean_r2 > best_r2:
            best_r2, best_params = mean_r2, {"alpha": alpha, "l1_ratio": l1}

best_model = train_model(best_params["alpha"], best_params["l1_ratio"], X_train_np, z_train)
print("Best params:", best_params)


In [ ]:
# save the model and parameters
with open('../data/processed/elasticnet_model.pkl', 'wb') as f:
    pickle.dump({'best_model': best_model, 'best_params': best_params}, f)

In [ ]:
# load the model and parameters
with open('../data/processed/elasticnet_model.pkl', 'rb') as f:
    saved = pickle.load(f)
    
best_model = saved['best_model']
best_params = saved['best_params']

In [ ]:
# ============================================================
# STEP 3a-iv â€” Reverse-transform predictions back to y space
# ============================================================

# --- Helper: reverse z_normal â†’ y ---

def inverse_transform(z_pred, skew, loc, scale, eps=1e-6):
    """
    Converts z_normal predictions back to the original y scale:
      1. Apply the normal CDF to z â†’ gets back a probability in [0,1].
      2. Clip to avoid edge issues (mirroring what we did during forward transform).
      3. Apply the PT3 PPF (inverse CDF) â†’ recovers the original y scale.
    """
    # Step 1: Normal CDF â€” "what percentile is each z-score?"
    p = norm.cdf(z_pred)

    # Step 2: Clip for numerical safety
    p = np.clip(p, eps, 1 - eps)

    # Step 3: PT3 PPF â€” "what y value sits at this percentile?"
    y_pred = pearson3.ppf(p, skew, loc=loc, scale=scale)
    return y_pred


# Predict z on the test set, then reverse-transform to original y scale.
def model_predict(model, X_np):
    X_t = torch.tensor(X_np, dtype=torch.float32).to(device)
    with torch.no_grad():
        return model(X_t).cpu().numpy().flatten()

z_pred_test  = model_predict(best_model, X_test_scaled.values)
z_pred_train = model_predict(best_model, X_train_scaled.values)
y_pred       = inverse_transform(z_pred_test, skew, loc, scale)
y_pred_train = inverse_transform(z_pred_train, skew, loc, scale)

In [ ]:
# ============================================================
# STEP 3a-v â€” Report metrics and scatterplot
# ============================================================

# --- Point estimates ---

r2_test  = r2_score(y_test, y_pred)
mse_test = mean_squared_error(y_test, y_pred)

print(f"\nTest RÂ²:  {r2_test:.4f}")
print(f"Test MSE: {mse_test:.4f}")

# --- Bootstrapped 95% CI for RÂ² (1,000 iterations) ---

np.random.seed(42)
n_boot = 1_000                            # number of bootstrap resamples
boot_r2 = np.empty(n_boot)               # will store one RÂ² per resample

y_test_arr = np.array(y_test)            # make sure we can index by integer array
y_pred_arr = np.array(y_pred)

for i in range(n_boot):
    # Draw n indices with replacement from the test set.
    idx = np.random.randint(0, len(y_test_arr), size=len(y_test_arr))
    boot_r2[i] = r2_score(y_test_arr[idx], y_pred_arr[idx])

# The 2.5th and 97.5th percentiles of the bootstrap distribution form the 95% CI.
ci_low, ci_high = np.percentile(boot_r2, [2.5, 97.5])

print(f"\nBootstrapped 95% CI for RÂ²: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Bootstrap mean RÂ²:          {boot_r2.mean():.4f}")

# --- Scatterplot: predicted vs actual ---

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_test, y_pred, alpha=0.4, s=20, color='steelblue', label='Test samples')

# Perfect-prediction reference line (y = x)
lims = [min(y_test_arr.min(), y_pred_arr.min()),
        max(y_test_arr.max(), y_pred_arr.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction (y = x)')

ax.set_xlabel('Actual y')
ax.set_ylabel('Predicted Å·')
ax.set_title(
    f'Predicted vs Actual  |  RÂ² = {r2_test:.3f}  |  95% CI [{ci_low:.3f}, {ci_high:.3f}]'
)
ax.legend()
plt.tight_layout()
plt.show()

# --- Interpretation notes (printed for reference) ---
print("""
Scatterplot interpretation:
  - Points clustered tightly around the red dashed line â†’ good predictions.
  - Systematic curves (fan shape, banana) â†’ remaining non-linearity or
    heteroskedasticity not captured by the linear model.
  - A wide CI means the RÂ² estimate is unstable â€” possibly due to small test set.
""")

In [ ]:
# ============================================================
# STEP 3a-vi â€” Per K-Means group RÂ²
# ============================================================

# 'kmeans_labels_test' should be a 1-D array/Series of integer cluster IDs
# aligned with X_test and y_test. Rename if yours is called something different.
# e.g. if you stored them as 'labels_test', replace the name below.

kmeans_labels_test = np.array(k_means_labels)   # ensure it is a numpy array

unique_labels = np.unique(k_means_labels)
print("\nPer-group RÂ² (K-Means clusters on test set):")
print("-" * 40)

group_r2 = {}   # store results for comparison

for label in unique_labels:
    mask = k_means_labels == label          # boolean mask for this cluster
    r2_group = r2_score(y_test_arr[mask], y_pred_arr[mask])
    group_r2[label] = r2_group
    n_group = mask.sum()
    print(f"  Cluster {label}  (n={n_group:4d})  RÂ² = {r2_group:.4f}")

print("-" * 40)
print(f"  Overall test RÂ²:          {r2_test:.4f}")

# --- Quick bar chart for visual comparison ---
fig2, ax2 = plt.subplots(figsize=(7, 4))
labels_sorted = sorted(group_r2.keys())
r2_vals = [group_r2[k] for k in labels_sorted]

ax2.bar([f'Cluster {k}' for k in labels_sorted], r2_vals,
        color='steelblue', alpha=0.75, edgecolor='white')
ax2.axhline(r2_test, color='red', linestyle='--', linewidth=1, label=f'Overall RÂ² = {r2_test:.3f}')
ax2.set_ylabel('RÂ²')
ax2.set_title('RÂ² by K-Means cluster (test set)')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 3a-vii â€” Non-zero coefficients and top 5 by magnitude
# ============================================================

# Retrieve the coefficients from the best ElasticNet model.
# Each coefficient corresponds to one input feature, telling us
# how much a one-unit change in that feature shifts z_normal (our transformed target).
weights = best_model.linear.weight.detach().cpu().numpy().flatten()
bias    = best_model.linear.bias.detach().cpu().numpy().item()

feature_names = X_train_scaled.columns.tolist()
coef_series = pd.Series(weights, index=feature_names)

# --- Non-zero coefficients ---
# ElasticNet's L1 penalty drives less useful coefficients exactly to zero.
# The ones that survive (non-zero) are the features the model found meaningful.
nonzero_coefs = coef_series[coef_series.abs() > 1e-4]
print(f"Total features:            {len(coef_series)}")
print(f"Non-zero coefficients:     {len(nonzero_coefs)}")
print(f"Zeroed-out (L1 excluded):  {len(coef_series) - len(nonzero_coefs)}")

# --- Top 5 by magnitude ---
# We sort by absolute value because a large negative coefficient is just as
# influential as a large positive one â€” both indicate strong relationships.
top5 = coef_series.reindex(coef_series.abs().sort_values(ascending=False).index).head(5)

print("\nTop 5 coefficients by magnitude:")
print("-" * 45)
for feat, val in top5.items():
    direction = "â†‘ positive" if val > 0 else "â†“ negative"
    print(f"  {feat:<30} {val:+.4f}  ({direction})")
print("-" * 45)

# --- Bar chart of top 5 ---
fig, ax = plt.subplots(figsize=(8, 4))

colors = ['steelblue' if v > 0 else 'coral' for v in top5.values]
ax.barh(top5.index[::-1], top5.values[::-1], color=colors[::-1], edgecolor='white')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient value (in z_normal space)')
ax.set_title('Top 5 ElasticNet coefficients by magnitude')
plt.tight_layout()
plt.show()

# --- All non-zero coefficients for reference ---
print("\nAll non-zero coefficients (sorted by magnitude):")
print(coef_series.reindex(coef_series.abs().sort_values(ascending=False).index)
      [coef_series.abs() > 1e-4].to_string())